## 17. LangGraph

<br>

### LangGraph 탄생 배경
1. **LLM이 생성한 답변이 환각이 아닌가?**
2. **RAG를 적용하여 받은 답변이 문서에는 없는 '사전지식'으로 답변한 것은 아닌가?**
3. **문서 검색에서 원하는 내용이 없는 경우, 인터넷 혹은 논문에서 부족한 정보를 검색하여 지식을 보강할 수는 없는가?**

<br>

### RAG 개발 단계의 고민 사례
- 일반 RAG를 수행했을 때 접하는 상황:
  - 문서 내 질문에 대한 답변이 존재하지 않음

    $\rightarrow$ 부족한 정보를 Web 검색하여 문서에 추가하는 로직을 추가

    $\rightarrow$ 검색결과에 잘못된 정보가 포함되거나 검색

    $\rightarrow$ 잘못된 검색결과가 환각으로 이어짐

    $\rightarrow$ 검색이 제대로 나올때까지 반복

    $\rightarrow$ 토큰 폭증 or 환각 방지 LLM 추가

    $\rightarrow$ **코드가 점점 길어지고 복잡해지며, 답변 품질 저하**

<br>

### Conventional RAG의 문제점
- 사전에 정의된 데이터 소싱 자원
- 사전에 정의된 Fixed Size Chunk
- 사전에 정의된 Query
- 사전에 정의된 검색방법
- 신뢰하기 어려운 LLM 혹은 Agent
- 고정된 프롬프트 형식
- LLM의 답변 결과에 대한 문서와의 관련성/신뢰성
  
<br>

- **RAG 파이프라인이 단방향 구조** : Document Loader(데이터 로드) $\rightarrow$ Answer (답변)
  - 이전 단계로 되돌아가기 어려우며, 이전 과정의 결과물을 수정하기 어려움
  
<br>

### LangGraph 제안
- 각 세부과정을 **노드(Node)** 라고 정의
- 이전 노드 $\rightarrow$ 다음 노드 : **엣지(Edge)** 연결
- **조건부 엣지**를 통해 분기 처리
  
  $\rightarrow$ RAG 파이프라인을 보다 유연하게 설계

<br>

#### LangGraph 구현 에시
- **기존 LangChain**

<img src='img/lg01.png' width=800>  

<br>

- **LangGraph**
  - 평가자 & QueryTransform 추가
  - 추가 검색기를 통한 문맥(Context) 보강
  - 문서-답변 간 관련성 여부를 판단하는 평가자 2를 추가하여 검증


<img src='img/lg02.png' width=800>

<br>

<img src='img/lg03.png' width=800>

<br>

### LangGraph의 특징
- Node (노드), Edge (엣지), State (상태관리)를 통해 LLM을 활용한 워크플로우에 순환(Cycle) 연산 기능을 추가하여 흐름을 제어
- RAG 파이프라인의 세부 단계별 흐름제어가 가능
- **Conditional Edge**: 조건부 (if, elif else 등) 흐름 제어
- **Human-in-the-loop** : 필요시 중간 개입하여 다음 단계를 결정
- **Checkpointer** : 과거 실행 과정에 대한 "수정" & "리플레이" 기능
  
<br>

#### 주요 용어
- Node (노드) : 어떤 작업 (Task)를 수행할지 정의
- Edge (엣지) : 다음으로 실행할 동작 정의
- State (상태) : 현재 상태 값을 저장 및 전달하는데 활용
- Conditional Edge (조건부 엣지) : 조건에 따라 분기 처리

<br>

<img src='img/lg04.png' width=800>

<br>

<hr>

<br>

## 상태 (State)
- **노드와 노드 간 정보를 전달할 때 상태(State) 객체에 담아 전달**
  - **`TypeDict`** : 일반 파이썬 `dict`에 타입힌팅을 추가한 개념
  - 모든 값을 다 채우지 않아도 됨
  - 새로운 노드에서 값을 덮어쓰기(Overwrite) 방식으로 채움

```python
from typing import TypeDict

class GraphState(TypeDict):
    goal: str
    todo: list[str]
    current_job: str
    total_time: int
    time_spent: int
    status: str

```

<br>

### 노드 별 상태 값의 변화
- 각 노드에서 새롭게 업데이트 하는 값은 기존 Key 값을 덮어쓰는 방식
- 노드에서 필요한 상태 값을 조회하여 동작에 활용할 수 있음
- 노드4 에서 노드 1에 반영된 LLM 값이 그대로 상태 전달되어 전달 가능

<img src='img/lg05.png' width=600>

<br>

#### 상태 (in RAG)
- 노드4에서 문서에 대하여 답변 관련성 점수를 부여
  - `score`가 BAD인 경우 $\rightarrow$
    - **노드1** : 질문을 재작성 (Query Transform)
    - **노드2** : 문서를 다시 검색 / 검색을 통한 정보 보완 (Context Retrieval 조정)
    - **노드3** : 답변을 재작성 요청 (프롬프트 조정, 다른 LLM 사용 등)
 
<img src='img/lg06.png' width=600>

<br>

<hr>

<br>

## 노드 (Node)
- 함수로 정의
- 입력인자 : 상태 (State)객체
- 반환
  - 대부분 상태 (State) 객체
  - Conditional Edge의 경우 다를 수 있음

<img src='img/lg07.png' width=600>

```python
def retrieve_document(state: GraphState) -> GraphState:

    retrieved_docs = pdf_retriever.invoke(state["questeion"])
    return GraphState(context=format_docs(retrieved_docs))

def llm_answer(state: GraphState) -> GraphState:

    return GraphState(
        answer=pdf_chain.invoke({"question": state["question"], "context":state["context"]})
    )

```

<br>

#### Graph 생성 후 노드 추가
- 이전에 정의한 함수를 Graph에 추가 (`add_node("노드이름" 함수)`)

```python
from langgraph.graph import END, StateGraph
from langgraph.checkpoint.memory import MemoryServer

workflow = StateGraph(GraphState)

# 노드 정의
workflow.add_node("retrieve", retrieve_document)
workflow.add_node("llm_answer", llm_answer)

```

<br>

<hr>

<br>

## 엣지 (Edge)
- 노드에서 노드간의 연결
- `add_edge("노드이름", "노드이름")`
  - from $\rightarrow$ to

```python
workflow.add_edge("retrieve", "llm_answer")
workflow.add_edge("llm_aswer", "relevance_check")
```

<img src='img/lg08.png' width=600>

<br>

### 조건부 엣지 (Conditional Edge)
- 노드에 조건부 엣지를 추가하여 분기를 수행
- `add_conditional_edges("노드이름", 조건부 판단 함수, dict 로 다음단계 결정)`

```python
workflow.add_conditional_edges(
    "relevance_check",
    is_relevant, # 조건부 판단 함수
    {
        "grounded": END,
        "notGrounded": "llm_answer",
        "notSure": "llm_answer"
    }
)

```

<br>

### 시작점 지정
- 지정한 시적점부터 Graph가 시작
  - `set_entry_point("노드이름")`

```python
workflow.set_entry_point("retrieve")
```

<br>

<img src='img/lg09.png' width=600>

<br>

<hr>

<br>


<br>

## 그래프 생성 및 시각화

<br>

### 체크포인터(memory)
- Checkpointer : 각 노드간 실행결과를 췆ㄱ하기 위한 메모리 (대화에 대한 기록과 유사)
- - 체크포인터를 활용해 특정 시점(Snapshot)으로 되돌리기 기능도 가능
- `compile(checkpointer=memory)` 지정하여 그래프 생성

```python
# 기록을 위한 메모리 저장
memory = MemorySaver()

# 그래프를 컴파일
app = workflow.compile(checkpointer=memory)

```


<br>

### 그래프 시각화
- `get_graph(xray=True).draw_mermaid_png()`

<br>

<hr>

<br>

## 실행 및 결과확인

<br>

### 그래프 실행
- `RunnableConfig`
  - `recursion_limit` : 최대 노드 실행 개수를 지정 (13인 경우, 총 13개의 노드까지 실행)
  - `thread_id` : 그래프 실행 아이디를 기록하고, 추후 추적하기 위한 목적으로 활용
- 상태 (State)로 시작
  - `question`에 질문만 입력하고, 상태를 첫 노드에게 전달

<br>

```python
from langchain_core.runnables import RunnableConfig

config = RunnableConfig(recursion_limit=13, configurable={"thread_id": "SELF-RAG"})

inputs = GraphState(question="삼성전자가 개발한 생성형 AI의 이름은?")
output = app.invoke(inputs, config=config)
```

<br>

### 결과 확인
- 출력된 결과로 최종 확인
- **출력 결과도 상태(State)에 담겨있음**

```python
print("Question: \t", output["question"])
print("Answer: \t", output["answer"])
print("Relevance: \t", output["relevance"])
```

<br>

<hr>

<br>

## Self-RAG
- https://arxiv.org/pdf/2310.11511
- **배경**
  - **Fixed Size Retrieval** : 좋든 싫든 정해진 크기만큼 검색하여 가져오기 떄문에, 검색에 노이즈가 존재
  - **무분별하게 주입되는 검색으로 인하여 문서내 다른 정보를 참고하거나, 제대로 된 답변이 나오지 않는 경우가 존재**
  - 검색된 정보의 정확성을 신뢰할 수 없음
- **제안**
  - 선택적 Retrieval 을 도입 (필요한 만큼만 Retriveal)
  - Retrieval로 부터 답변을 도출
  - **도출된 답변과 Retrieval Passage 간 관련성 체크**

<br>

<img src='img/lg10.png' width=800>

<br>

<img src='img/lg11.png' width=200>

<br>

<hr>

<br>

## Corrective-RAG
- https://arxiv.org/pdf/2401.15884
- 배경
  - RAG의 답변 결과는 검색된 문서의 관련성에 크게 의존적 $\rightarrow$ 검색이 잘못된 경우에 답변에 대한 품질 저하가 우려
- 제안
  - 사용자 입력 쿼리에 대하여 검색된 문서의 품질을 평가 $\rightarrow$ 검색된 결과가 사용자 입력 쿼리와 관련성이 높도록 쿼리를 수정 (Corrective)

<br>

<img src='img/lg12.png' width=600>

<br>

### Corrective-RAG 구현 예시

- **전통적인 RAG를 수행하되, 검색된 문서에 답변에 필요한 정보가 부족한 경우 "웹 검색"을 위한 쿼리 재작성**

  $\rightarrow$ 웹 검색으로 보충된 정보로 답변 도출 시도

  $\rightarrow$ 새로운 답변으로 관련성 체크 후 재조정/종료 진행 

<br>

<img src='img/lg13.png' with=200>

<br>

#### 1. `__start__` : 질문 입력
- `생성형 AI 가우스를 만든 회사의 2023년도 매출액은 얼마인가요?`

<br>

#### 2. `retrieve` : PDF 문서에 대한 검색 수행

<img src='img/lg14.png' width=500>

<br>

#### 3. `llm_answer`: 문서 기반 답변 도출

```
생성현 AI '가우스'를 만든 회사는 삼성전자입니다.
그러나 제공된 문맥에서는 삼성전자의 2023년 매출액에 대한 정보는 언급되어있지 않습니다.
따라서 삼성전자의 2023년도 매출액에 대한 정보를 제공할 수 없습니다.
```

#### 4. `relevance_check` : 관련성 / 유효성 체크
- **직전 답변의 관련성/유효성 $\rightarrow$ `"notGrounded"`**

<br>

#### 5. `rewrite` : 추가정보 검색을 위한 쿼리 재작성

<br>

#### 6. `search_on_web` : 재작성된 쿼리로 검색 수행
- `"삼성전자의 2023년도 매출액은 얼마 인가요?"`

<img src='img/lg16.png' width=400>

<br>

#### 7. `llm_answer` : 문서 기반 답변 도출

```
삼성전자의 2023년도 매출액은 258.94조원 입니다.
(출처: https://news........)
```

<br>

#### 8. `relevance_check` : 관련성 / 유효성 체크
- **직전 답변의 관련성/유효성 $\rightarrow$ `"grounded"`**

<br>

#### 9. `__end__` : 종료
- 질문: `생성형 AI 가우스를 만든ㄷ 회사의 2023년도 매출액은 얼마인가요?`
- 답변1 : `삼성전자의 2023년도 매출액은 258.94조원입니다.`
- 답변2 : `생성형 AI "가우스"를 만든 회사는 삼성전자이며, 2023년도 매출액은 258.94조원 입니다.`

<br>

<hr>